# Using .env to Keep Database Credentials Secret in Jupyter

This notebook is part of issue #14 (Using `.env` to Keep Database Credentials Secret in Jupyter).
The goal is to load PostgreSQL credentials from a `.env` file with `python-dotenv` instead of
hardcoding them anywhere in the notebook, then use them to attempt a database connection.

## Step 1: Install python-dotenv

Installed once via `pip install python-dotenv` (and `psycopg2-binary` to actually connect to Postgres).
Not re-run here since it only needs to happen once per environment.

## Step 2: Create a .env file

A `.env` file sits in the project root with the database credentials as key/value pairs
(`DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD`). It is never committed — see Step 4.
A `.env.example` file with placeholder values is committed instead, so anyone cloning the repo
knows which variables to set.

## Step 3: Load the credentials with python-dotenv

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")

# Never print the raw password, only confirm it was loaded and mask it
masked_password = (db_password[:2] + "***") if db_password else None

print(f"DB_HOST: {db_host}")
print(f"DB_PORT: {db_port}")
print(f"DB_NAME: {db_name}")
print(f"DB_USER: {db_user}")
print(f"DB_PASSWORD: {masked_password}")

DB_HOST: localhost
DB_PORT: 5432
DB_NAME: focusbear_dev
DB_USER: focusbear_user
DB_PASSWORD: lo***


## Step 4: Confirm .env is excluded from version control

In [2]:
with open("../.gitignore") if not os.path.exists(".gitignore") else open(".gitignore") as f:
    gitignore_contents = f.read()

print(".env is gitignored:", ".env" in gitignore_contents.splitlines())

.env is gitignored: True


## Step 5: Use the credentials to connect to PostgreSQL

The credentials loaded from `.env` are used directly to build the connection, they're never
typed into the notebook itself. There's no local Postgres server running in this environment,
so the connection is expected to fail, but it proves the credentials came from `.env` and were
actually used rather than just printed.

In [3]:
import psycopg2

try:
    conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        dbname=db_name,
        user=db_user,
        password=db_password,
        connect_timeout=3,
    )
    print("Connected successfully.")
    conn.close()
except psycopg2.OperationalError as e:
    print("Connection attempt failed (expected, no local Postgres server running):")
    print(e)

Connection attempt failed (expected, no local Postgres server running):
connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?



## Insights

- The notebook never contains a real credential value, only variable names pulled from the
  environment. Anyone reading or sharing this notebook (or its output) can't see the password.
- Because `.env` is gitignored, credentials can't accidentally end up in a commit or a public
  repo, while `.env.example` still documents what variables are needed to run the notebook.
- `load_dotenv()` plus `os.getenv()` is the entire integration, one import and one function
  call before every credential lookup becomes a normal environment variable read.